In [1]:
"""
================================================================================
MEMORY-OPTIMIZED ENHANCED AI-NIDS IMPLEMENTATION
With SOTA Comparisons & Deep Error Analysis
================================================================================

FIXES APPLIED:
1. optimize_dataframe_dtypes(): fixed duplicate column names and Series check.
2. AdvancedImbalanceHandler.fit_resample(): fixed .iloc indexing for large datasets.
3. UNSW-NB15 now selects the correct original CSV file (49 columns, with 'attack_cat')
   to avoid column mismatches from pre‑processed copies.

IMPROVEMENTS (v2.2 – targeted for NSL-KDD & CIC-ToN-IoT):
4. Categorical features (e.g., protocol_type, service, flag) are now safely encoded:
   - One‑hot (drop_first=True) for columns with ≤50 unique values.
   - Frequency encoding for high‑cardinality columns (>50 unique values).
   This avoids memory blow‑up while retaining discriminative information.
5. The binary filter in Phase 1 now correctly assumes index 0 = normal class.
   A custom label encoding guarantees that the “normal”/“benign” class
   (case‑insensitive) is always mapped to 0, aligning the filter’s (y != 0) logic
   with the data.
6. Dataset‑specific preprocessing for UNSW‑NB15 and CIC‑ToN‑IoT modified to
   preserve categorical columns for the unified encoding step.

Author: Enhanced Research Implementation
Date: 2026-08-05
================================================================================
"""

# ============================================================================
# IMPORTS
# ============================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Core ML
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    matthews_corrcoef, cohen_kappa_score, balanced_accuracy_score
)
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, SelectKBest

# Advanced Imbalance
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.combine import SMOTEENN, SMOTETomek
from collections import Counter

# XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

# Deep Learning
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models, callbacks
    TF_AVAILABLE = True
except ImportError:
    TF_AVAILABLE = False

# Utilities
import shap
import gc
import os
import time
import json
import logging
from tqdm import tqdm
from datetime import datetime
import psutil
import warnings

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"✅ SHAP version: {shap.__version__}")

# ============================================================================
# CONFIGURATION
# ============================================================================

CONFIG = {
    'sampling_rate': 0.15,  # STRENGTHENED (v2.1): was 0.1
    'mrmr_features': 30,    # STRENGTHENED (v2.1): was 20
    'model_type': 'random_forest',
    'use_smote': True,
    'smote_type': 'BorderlineSMOTE',  # STRENGTHENED (v2.1): was 'SMOTE'
    'leakage_threshold': 0.9,
    'explainer_type': 'tree',
    'test_size': 0.2,
    'random_seed': 42,
    'n_folds': 3,
    'shap_samples': 50,  # STRENGTHENED (v2.1): was 30
    'output_dir': '/kaggle/working/results/',
    'deep_learning_epochs': 20,
    'deep_learning_batch_size': 64,
    'max_rows': 500000,
    'chunk_size': 100000,
    'memory_threshold': 0.85,
    'max_categories_for_ohe': 50,    # NEW: threshold for one‑hot vs. frequency encoding
}

# Dataset configurations
DATASETS = {
    'NSL-KDD': {
        'path': '/kaggle/input/datasets/karthikragavenderb/nsl-kdd-bert',
        'file_patterns': ['KDD', 'kdd', 'nsl-kdd', 'NSL-KDD', '.csv', '.data'],
        'label_col': 'Label',
        'description': 'Benchmark dataset',
        'skip_files': ['.jpg', '.png', '.jpeg'],
        'max_rows': 200000
    },
    'UNSW-NB15': {
        'path': '/kaggle/input/datasets/likkisamarthreddy/unsw-nb15',
        'file_patterns': ['UNSW', 'unsw', 'UNSW-NB15', '.csv'],
        'label_col': 'Label',
        'description': 'Modern attack coverage',
        'max_rows': 300000
    },
    'CIC-IDS2017': {
        'path': '/kaggle/input/datasets/bertvankeulen/cicids-2017',
        'file_patterns': ['CIC-IDS2017', 'cicids2017', '.csv'],
        'label_col': 'Label',
        'description': 'Primary dataset with extreme class imbalance',
        'max_rows': 300000
    },
    'CSE-CICIDS2018': {
        'path': '/kaggle/input/datasets/shrey213/cicids2018',
        'file_patterns': ['CSE-CICIDS2018', 'csecicids2018', '.parquet'],
        'label_col': 'Label',
        'description': 'Large-scale validation',
        'max_rows': 300000
    },
    'CIC-ToN-IoT': {
        'path': '/kaggle/input/datasets/wahidulislambayazid/cic-ton-iot',
        'file_patterns': ['ToN-IoT', 'toniot', 'CIC-ToN-IoT', '.parquet', '.csv'],
        'label_col': 'Label',
        'description': 'IoT-specific testing',
        'max_rows': 300000
    }
}

# ============================================================================
# MEMORY UTILITY FUNCTIONS
# ============================================================================

def get_memory_usage():
    """Get current memory usage in GB."""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024 / 1024

def check_memory(threshold=0.85):
    """Check if memory usage exceeds threshold."""
    memory_gb = get_memory_usage()
    total_memory = psutil.virtual_memory().total / 1024 / 1024 / 1024
    usage_ratio = memory_gb / total_memory
    return usage_ratio < threshold, usage_ratio

# ============================================================================
# FIXED: optimize_dataframe_dtypes()
# ============================================================================

def optimize_dataframe_dtypes(df):
    """Optimize DataFrame memory usage by downcasting dtypes."""
    # Ensure column names are unique
    if not df.columns.is_unique:
        cols = pd.Series(df.columns)
        for dup in cols[cols.duplicated()].unique():
            dup_indices = cols[cols == dup].index
            for i, idx in enumerate(dup_indices):
                if i > 0:
                    cols.iloc[idx] = f"{dup}_{i}"
        df.columns = cols

    for col in df.columns:
        if not isinstance(df[col], pd.Series):
            continue
        col_type = df[col].dtype
        if col_type != 'object':
            try:
                if col_type.kind in ['i', 'u']:
                    c_min = df[col].min()
                    c_max = df[col].max()
                    if c_min >= 0:
                        if c_max < 255:
                            df[col] = df[col].astype('uint8')
                        elif c_max < 65535:
                            df[col] = df[col].astype('uint16')
                        elif c_max < 4294967295:
                            df[col] = df[col].astype('uint32')
                        else:
                            df[col] = df[col].astype('uint64')
                    else:
                        if c_min > -128 and c_max < 127:
                            df[col] = df[col].astype('int8')
                        elif c_min > -32768 and c_max < 32767:
                            df[col] = df[col].astype('int16')
                        elif c_min > -2147483648 and c_max < 2147483647:
                            df[col] = df[col].astype('int32')
                        else:
                            df[col] = df[col].astype('int64')
                elif col_type.kind == 'f':
                    df[col] = df[col].astype('float32')
            except Exception:
                pass
    return df

def cleanup_memory():
    """Force garbage collection and memory cleanup."""
    gc.collect()
    if TF_AVAILABLE:
        tf.keras.backend.clear_session()
    print(f"   Memory after cleanup: {get_memory_usage():.2f} GB")

# ============================================================================
# DATA PREPROCESSOR (Memory Optimized) – v2.2 improvements integrated
# ============================================================================

class MemoryOptimizedDataPreprocessor:
    def __init__(self, dataset_name, leakage_threshold=0.9):
        self.dataset_name = dataset_name
        self.leakage_threshold = leakage_threshold
        self.leaky_features = []
        self.kept_features = []
        self.label_encoder = LabelEncoder()
        self.preprocessing_time = 0
        self.original_columns = []
        self.label_col = 'Label'
        self.freq_encodings = {}  # store frequency mappings for high‑cardinality columns
        
    def load_data_from_kaggle(self, dataset_config):
        print(f"📂 Loading {self.dataset_name}...")
        start_time = time.time()
        
        dataset_path = dataset_config['path']
        skip_files = dataset_config.get('skip_files', [])
        max_rows = dataset_config.get('max_rows', CONFIG['max_rows'])
        
        if not os.path.exists(dataset_path):
            print(f"   ❌ Path not found: {dataset_path}")
            return None
        
        # Find files
        all_files = []
        for root, dirs, files in os.walk(dataset_path):
            for file in files:
                all_files.append(os.path.join(root, file))
        
        # Filter out images
        filtered_files = []
        for f in all_files:
            f_lower = f.lower()
            skip = False
            for skip_pattern in skip_files:
                if skip_pattern in f_lower:
                    skip = True
                    break
            if not skip:
                filtered_files.append(f)
        
        if not filtered_files:
            print(f"   ❌ No suitable files found")
            return None
        
        print(f"   Found {len(filtered_files)} files")
        
        # ============================================================
        # SPECIAL HANDLING FOR UNSW-NB15
        # ============================================================
        if self.dataset_name == 'UNSW-NB15':
            import re
            pattern = re.compile(r'unsw-nb15', re.IGNORECASE)
            csv_files = [f for f in filtered_files
                         if f.lower().endswith('.csv') and pattern.search(f)]
            if not csv_files:
                print("   ❌ No UNSW-NB15 CSV files found")
                return None

            chosen_file = None
            for f in csv_files:
                try:
                    sample = pd.read_csv(f, nrows=0, encoding='latin-1')
                    cols = [c.strip().lower().replace(' ', '_') for c in sample.columns]
                    has_label = ('attack_cat' in cols or 'label' in cols)
                    if has_label and len(cols) < 100:
                        chosen_file = f
                        break
                except:
                    continue

            if chosen_file is None:
                chosen_file = max(csv_files, key=lambda x: os.path.getsize(x))
                print(f"   ⚠️ No standard UNSW‑NB15 file found, using largest: {os.path.basename(chosen_file)}")
            else:
                print(f"   Selected standard file: {os.path.basename(chosen_file)}")

            df = self._load_file_memory_optimized(chosen_file, max_rows)
            if df is None or len(df) == 0:
                return None

            df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
            df.columns = df.columns.str.replace('[^a-zA-Z0-9_]', '', regex=True)
            df = df.replace([np.inf, -np.inf], np.nan)
            df = optimize_dataframe_dtypes(df)
            df = self._dataset_specific_preprocessing(df)
            self.original_columns = df.columns.tolist()
            elapsed = time.time() - start_time
            print(f"   ✅ Loaded {len(df):,} rows, {len(df.columns)} columns in {elapsed:.2f}s")
            print(f"   Memory: {get_memory_usage():.2f} GB")
            return df
        # ============================================================
        
        selected_file = self._select_best_file(filtered_files, dataset_config.get('file_patterns', []))
        
        if selected_file is None:
            return None
        
        file_size_mb = os.path.getsize(selected_file) / (1024 * 1024)
        print(f"   Selected: {os.path.basename(selected_file)} ({file_size_mb:.2f} MB)")
        
        try:
            df = self._load_file_memory_optimized(selected_file, max_rows)
            if df is None or len(df) == 0:
                print(f"   ❌ Failed to load data")
                return None
            
            df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
            df.columns = df.columns.str.replace('[^a-zA-Z0-9_]', '', regex=True)
            df = df.replace([np.inf, -np.inf], np.nan)
            df = optimize_dataframe_dtypes(df)
            df = self._dataset_specific_preprocessing(df)
            
            self.original_columns = df.columns.tolist()
            
            elapsed = time.time() - start_time
            print(f"   ✅ Loaded {len(df):,} rows, {len(df.columns)} columns in {elapsed:.2f}s")
            print(f"   Memory: {get_memory_usage():.2f} GB")
            return df
            
        except Exception as e:
            print(f"   ❌ Error loading data: {e}")
            import traceback
            traceback.print_exc()
            return None
    
    def _load_file_memory_optimized(self, file_path, max_rows):
        try:
            if file_path.endswith('.parquet'):
                try:
                    df = pd.read_parquet(file_path, engine='pyarrow')
                except:
                    df = pd.read_parquet(file_path)
            elif file_path.endswith('.csv'):
                file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
                if file_size_mb > 500:
                    print(f"   ⚠️ Large CSV ({file_size_mb:.1f} MB), using chunked reading...")
                    chunks = []
                    n_chunks = 0
                    for encoding in ['utf-8', 'latin-1']:
                        try:
                            for chunk in pd.read_csv(file_path, chunksize=CONFIG['chunk_size'],
                                                     low_memory=False, encoding=encoding):
                                chunks.append(chunk)
                                n_chunks += 1
                                if max_rows is not None and n_chunks * CONFIG['chunk_size'] >= max_rows:
                                    break
                            df = pd.concat(chunks, ignore_index=True)
                            break
                        except (UnicodeDecodeError, pd.errors.ParserError):
                            if encoding == 'latin-1':
                                raise
                            continue
                else:
                    for encoding in ['utf-8', 'latin-1']:
                        try:
                            df = pd.read_csv(file_path, low_memory=False, encoding=encoding)
                            break
                        except (UnicodeDecodeError, pd.errors.ParserError):
                            if encoding == 'latin-1':
                                raise
                            continue
            elif file_path.endswith(('.txt', '.data')):
                try:
                    df = pd.read_csv(file_path, low_memory=False)
                except:
                    df = pd.read_csv(file_path, header=None, low_memory=False)
                    if len(df.columns) == 42:
                        df.columns = [f'feature_{i}' for i in range(len(df.columns) - 1)] + ['Label']
            else:
                df = pd.read_csv(file_path, low_memory=False)

            if max_rows is not None and len(df) > max_rows:
                print(f"   ⚠️ Dataset has {len(df):,} rows, sampling to {max_rows:,}")
                df = df.sample(n=max_rows, random_state=42)
            return df
        except Exception as e:
            print(f"   ❌ Error loading file: {e}")
            return None
    
    def _select_best_file(self, files, patterns):
        matching_files = []
        for f in files:
            f_lower = f.lower()
            if any(p.lower() in f_lower for p in patterns):
                matching_files.append(f)
        if matching_files:
            selected = max(matching_files, key=lambda x: os.path.getsize(x))
            print(f"   Found {len(matching_files)} matching files")
            return selected
        data_files = [f for f in files if f.endswith(('.csv', '.parquet', '.feather', '.data', '.txt'))]
        if data_files:
            selected = max(data_files, key=lambda x: os.path.getsize(x))
            print(f"   Found {len(data_files)} data files")
            return selected
        return files[0] if files else None
    
    def _dataset_specific_preprocessing(self, df):
        if self.dataset_name == 'UNSW-NB15':
            print("   Applying UNSW-NB15 specific preprocessing...")
            label_candidates = ['attack_cat', 'label', 'class', 'Label', 'Attack_cat']
            self.label_col = None
            for col in label_candidates:
                if col in df.columns:
                    self.label_col = col
                    break
            if self.label_col is None:
                self.label_col = self.identify_label_column(df)

            # ✅ REMOVED the aggressive pd.to_numeric conversion that destroyed
            # categorical columns (proto, service, state).
            n_unique_labels = df[self.label_col].nunique(dropna=True)
            if n_unique_labels < 2:
                raise ValueError(
                    f"UNSW-NB15 label column '{self.label_col}' has only "
                    f"{n_unique_labels} unique value(s) after preprocessing."
                )

        elif self.dataset_name == 'NSL-KDD':
            print("   Applying NSL-KDD specific preprocessing...")
            if 'label' in df.columns:
                self.label_col = 'label'
            elif 'class' in df.columns:
                self.label_col = 'class'
            elif 'Label' in df.columns:
                self.label_col = 'Label'

        elif self.dataset_name == 'CIC-ToN-IoT':
            print("   Applying ToN-IoT specific preprocessing...")
            label_candidates = ['label', 'Label', 'attack', 'Attack', 'class', 'Class']
            for col in label_candidates:
                if col in df.columns:
                    self.label_col = col
                    break
            # ✅ Preserve categorical columns; only convert numeric-like columns.
            for col in df.columns:
                if df[col].dtype == 'object':
                    continue  # preserve categorical columns
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    pass
        return df
    
    def identify_label_column(self, df):
        possible_labels = ['label', 'class', 'attack_cat', 'attack_type', 'category', 'normal', 'Label', 'Class', 'Attack_Cat']
        for col in df.columns:
            col_lower = col.lower()
            if any(pat.lower() in col_lower for pat in possible_labels):
                return col
        for col in df.columns:
            try:
                if df[col].nunique() <= 5 and df[col].dtype in ['int64', 'float64', 'int32']:
                    return col
            except:
                pass
        return df.columns[-1]
    
    # ✅ Memory‑efficient categorical encoding
    def _encode_categorical_features(self, X):
        obj_cols = X.select_dtypes(include=['object']).columns.tolist()
        if not obj_cols:
            return X
        
        X_enc = X.copy()
        for col in obj_cols:
            n_unique = X_enc[col].nunique(dropna=False)
            if n_unique <= CONFIG['max_categories_for_ohe']:
                # One‑hot encoding, drop first to reduce redundancy
                dummies = pd.get_dummies(X_enc[col], prefix=col, drop_first=True)
                X_enc = pd.concat([X_enc.drop(columns=[col]), dummies], axis=1)
            else:
                # Frequency encoding – replace category with its count
                freq_map = X_enc[col].value_counts().to_dict()
                self.freq_encodings[col] = freq_map
                X_enc[col] = X_enc[col].map(freq_map).astype('float32')
        return X_enc
    
    def identify_leaky_features(self, df, label_col='Label'):
        print("🔍 Identifying leaky features...")
        start_time = time.time()
        
        if label_col not in df.columns:
            label_col = self.identify_label_column(df)
            if label_col is None:
                raise ValueError("Label column not found")
        self.label_col = label_col
        print(f"   Using label column: '{label_col}'")
        
        X = df.drop(columns=[label_col])
        y = df[label_col].astype(str)

        # ✅ Encode categorical features BEFORE leakage detection
        X = self._encode_categorical_features(X)

        n_unique_labels = y.nunique(dropna=True)
        if n_unique_labels < 2:
            raise ValueError(
                f"Label column '{label_col}' for dataset '{self.dataset_name}' "
                f"has only {n_unique_labels} unique value(s). Refusing to "
                f"train on a degenerate label."
            )

        try:
            y_encoded = self.label_encoder.fit_transform(y)
        except:
            y_encoded = pd.factorize(y)[0]
        
        # After encoding, all columns should be numeric; convert any remaining
        # non‑numeric to float (errors='coerce') and drop all‑NaN columns.
        for col in X.columns:
            try:
                if X[col].dtype == 'object':
                    X[col] = pd.to_numeric(X[col], errors='coerce')
            except:
                pass
        X = X.dropna(axis=1, how='all')
        
        correlations = {}
        for col in X.columns:
            try:
                if X[col].nunique() <= 1:
                    continue
                valid_mask = ~pd.isna(X[col])
                if valid_mask.sum() > 10:
                    subset_size = min(10000, len(X))
                    if len(X) > subset_size:
                        idx = np.random.choice(len(X), subset_size, replace=False)
                        x_subset = X[col].values.astype(float)[idx]
                        y_subset = y_encoded[idx]
                    else:
                        x_subset = X[col].values.astype(float)[valid_mask]
                        y_subset = y_encoded[valid_mask]
                    corr = np.corrcoef(x_subset, y_subset)[0, 1]
                    if not np.isnan(corr):
                        correlations[col] = abs(corr)
            except:
                continue
        
        self.leaky_features = [col for col, corr in correlations.items() if corr > self.leakage_threshold]
        leak_patterns = ['time', 'date', 'timestamp', 'duration', 'flow_duration', 
                        'id', 'index', 'seq', 'num', 'flow_id']
        for col in X.columns:
            col_lower = col.lower()
            if any(p in col_lower for p in leak_patterns):
                if col not in self.leaky_features:
                    try:
                        if X[col].nunique() == len(X):
                            self.leaky_features.append(col)
                    except:
                        pass
        
        self.kept_features = [col for col in X.columns if col not in self.leaky_features]
        elapsed = time.time() - start_time
        print(f"   ✅ Found {len(self.leaky_features)} leaky features")
        print(f"   ✅ Keeping {len(self.kept_features)} features")
        return X[self.kept_features], y, label_col
    
    def preprocess(self, df, label_col=None, test_mode=False):
        print("🔧 Preprocessing data...")
        start_time = time.time()
        
        df = df.copy()
        df = df.dropna(how='all')
        df = df.replace([np.inf, -np.inf], np.nan)
        
        if label_col is None or label_col not in df.columns:
            label_col = self.identify_label_column(df)
        
        X, y_str, label_col = self.identify_leaky_features(df, label_col)
        
        # ✅ Robust normal‑class detection: recognise 'normal' or 'benign' (case‑insensitive)
        normal_mask = y_str.str.lower().isin(['normal', 'benign'])
        if normal_mask.any():
            # Use the exact string for the normal class to preserve original casing
            normal_class = y_str[normal_mask].iloc[0]
            all_classes = [normal_class] + sorted([c for c in y_str.unique() if c != normal_class])
            class_to_idx = {cls: i for i, cls in enumerate(all_classes)}
            y_encoded = y_str.map(class_to_idx).values
        else:
            # Fallback to LabelEncoder (with a warning)
            print("   ⚠️ No 'normal'/'benign' class found – falling back to LabelEncoder.")
            if not test_mode:
                y_encoded = self.label_encoder.fit_transform(y_str)
            else:
                y_encoded = self.label_encoder.transform(y_str)
        
        if len(X) == 0:
            print("   ❌ No features remaining")
            return pd.DataFrame(), pd.Series()
        
        try:
            selector = VarianceThreshold(threshold=0.01)
            X_selected = selector.fit_transform(X)
            X_columns = X.columns[selector.get_support()].tolist()
        except Exception as e:
            print(f"   ⚠️ Variance threshold failed: {e}")
            X_selected = X.values
            X_columns = X.columns.tolist()
        
        X_final = pd.DataFrame(X_selected, columns=X_columns)
        for col in X_final.columns:
            try:
                X_final[col] = pd.to_numeric(X_final[col], errors='coerce')
            except:
                pass
        X_final = X_final.dropna(axis=1, how='all')
        X_final = X_final.fillna(0)
        X_final = optimize_dataframe_dtypes(X_final)
        
        elapsed = time.time() - start_time
        self.preprocessing_time = elapsed
        print(f"   ✅ Preprocessed: {len(X_final):,} rows, {len(X_final.columns)} features")
        print(f"   ✅ Preprocessing took {elapsed:.2f}s")
        
        del df
        cleanup_memory()
        return X_final, pd.Series(y_encoded, name='Label')
    
    def get_leakage_report(self):
        return {
            'dataset': self.dataset_name,
            'leaky_features': self.leaky_features,
            'leaky_count': len(self.leaky_features),
            'kept_count': len(self.kept_features),
            'original_columns': self.original_columns[:10],
            'preprocessing_time': self.preprocessing_time,
            'label_col': self.label_col
        }

# ============================================================================
# ADVANCED IMBALANCE HANDLING (FIXED)
# ============================================================================

class AdvancedImbalanceHandler:
    def __init__(self, technique='SMOTE', random_seed=42):
        self.technique = technique
        self.random_seed = random_seed
        self.sampler = None
        self.original_distribution = None
        self.resampled_distribution = None
        
    def get_sampler(self):
        if self.technique == 'SMOTE':
            return SMOTE(random_state=self.random_seed)
        elif self.technique == 'ADASYN':
            return ADASYN(random_state=self.random_seed)
        elif self.technique == 'BorderlineSMOTE':
            return BorderlineSMOTE(random_state=self.random_seed)
        elif self.technique == 'SMOTEENN':
            return SMOTEENN(random_state=self.random_seed)
        elif self.technique == 'SMOTETomek':
            return SMOTETomek(random_state=self.random_seed)
        else:
            return None
    
    def fit_resample(self, X, y):
        print(f"📊 Applying {self.technique} for class imbalance...")
        self.original_distribution = Counter(y)
        sampler = self.get_sampler()
        if sampler is None:
            print(f"   ⚠️ Unknown technique: {self.technique}")
            return X, y

        min_class_count = min(self.original_distribution.values())
        max_class_count = max(self.original_distribution.values())
        if max_class_count / min_class_count < 2:
            print(f"   ℹ️ Imbalance ratio {max_class_count/min_class_count:.2f} < 2, skipping")
            return X, y

        try:
            if min_class_count < 2:
                print(f"   ⚠️ Minimum class has {min_class_count} samples, skipping")
                return X, y

            if len(X) > 200000:
                print(f"   ⚠️ Large dataset ({len(X):,}), limiting SMOTE")
                sample_size = min(100000, len(X))
                idx = np.random.choice(len(X), sample_size, replace=False)
                X_subset = X.iloc[idx]
                y_subset = y.iloc[idx]
                X_resampled, y_resampled = sampler.fit_resample(X_subset, y_subset)
                remaining_idx = np.setdiff1d(np.arange(len(X)), idx)
                X_resampled = np.vstack([X_resampled, X.iloc[remaining_idx].values])
                y_resampled = np.concatenate([y_resampled, y.iloc[remaining_idx].values])
            else:
                X_resampled, y_resampled = sampler.fit_resample(X, y)

            self.resampled_distribution = Counter(y_resampled)
            print(f"   Original: {dict(self.original_distribution)}")
            print(f"   Resampled: {dict(self.resampled_distribution)}")
            return X_resampled, y_resampled

        except Exception as e:
            print(f"   ⚠️ Resampling failed: {e}")
            return X, y

# ============================================================================
# SOTA BASELINE MODELS (Memory Optimized)
# ============================================================================

class SOTABaselines:
    def __init__(self, random_seed=42):
        self.random_seed = random_seed
        self.results = {}
        self.training_times = {}
        
    def train_xgboost(self, X_train, y_train, X_test, y_test):
        print("   Training XGBoost...")
        start_time = time.time()
        if not XGB_AVAILABLE:
            return {'error': 'XGBoost not available'}
        try:
            class_counts = Counter(y_train)
            total = sum(class_counts.values())
            subsample = 0.8 if len(X_train) > 200000 else 1.0
            model = xgb.XGBClassifier(
                n_estimators=200,
                max_depth=8,
                learning_rate=0.08,
                random_state=self.random_seed,
                use_label_encoder=False,
                eval_metric='logloss',
                n_jobs=-1,
                subsample=subsample,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                min_child_weight=2
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            elapsed = time.time() - start_time
            self.training_times['xgboost'] = elapsed
            return self._compute_metrics(y_test, y_pred, model, X_test)
        except Exception as e:
            return {'error': str(e)}
    
    def train_naive_bayes(self, X_train, y_train, X_test, y_test):
        print("   Training Naive Bayes...")
        start_time = time.time()
        try:
            model = GaussianNB()
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            elapsed = time.time() - start_time
            self.training_times['naive_bayes'] = elapsed
            return self._compute_metrics(y_test, y_pred, model, X_test)
        except Exception as e:
            return {'error': str(e)}
    
    def train_dnn(self, X_train, y_train, X_test, y_test, n_classes):
        print("   Training DNN...")
        start_time = time.time()
        if not TF_AVAILABLE:
            return {'error': 'TensorFlow not available'}
        try:
            if len(X_train) > 100000:
                idx = np.random.choice(len(X_train), 100000, replace=False)
                X_train_sub = X_train[idx]
                y_train_sub = y_train[idx]
            else:
                X_train_sub = X_train
                y_train_sub = y_train
            model = keras.Sequential([
                layers.Input(shape=(X_train.shape[1],)),
                layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
                layers.Dropout(0.3),
                layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
                layers.Dropout(0.3),
                layers.Dense(n_classes, activation='softmax')
            ])
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=0.001),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
            early_stop = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
            history = model.fit(
                X_train_sub, y_train_sub,
                validation_split=0.2,
                epochs=CONFIG['deep_learning_epochs'],
                batch_size=CONFIG['deep_learning_batch_size'],
                callbacks=[early_stop],
                verbose=0
            )
            y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
            elapsed = time.time() - start_time
            self.training_times['dnn'] = elapsed
            return self._compute_metrics(y_test, y_pred, model, X_test)
        except Exception as e:
            return {'error': str(e)}
    
    def train_lstm(self, X_train, y_train, X_test, y_test, n_classes):
        print("   Training LSTM...")
        start_time = time.time()
        if not TF_AVAILABLE:
            return {'error': 'TensorFlow not available'}
        try:
            if len(X_train) > 50000:
                idx = np.random.choice(len(X_train), 50000, replace=False)
                X_train_sub = X_train[idx]
                y_train_sub = y_train[idx]
            else:
                X_train_sub = X_train
                y_train_sub = y_train
            X_train_seq = X_train_sub.reshape(X_train_sub.shape[0], X_train_sub.shape[1], 1)
            X_test_seq = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
            model = keras.Sequential([
                layers.LSTM(32, return_sequences=True, input_shape=(X_train.shape[1], 1)),
                layers.Dropout(0.3),
                layers.LSTM(16),
                layers.Dropout(0.3),
                layers.Dense(8, activation='relu'),
                layers.Dense(n_classes, activation='softmax')
            ])
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=0.001),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
            early_stop = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
            history = model.fit(
                X_train_seq, y_train_sub,
                validation_split=0.2,
                epochs=10,
                batch_size=CONFIG['deep_learning_batch_size'],
                callbacks=[early_stop],
                verbose=0
            )
            y_pred = np.argmax(model.predict(X_test_seq, verbose=0), axis=1)
            elapsed = time.time() - start_time
            self.training_times['lstm'] = elapsed
            return self._compute_metrics(y_test, y_pred, model, X_test)
        except Exception as e:
            return {'error': str(e)}
    
    def train_cnn_lstm(self, X_train, y_train, X_test, y_test, n_classes):
        print("   Training CNN-LSTM...")
        start_time = time.time()
        if not TF_AVAILABLE:
            return {'error': 'TensorFlow not available'}
        try:
            if len(X_train) > 50000:
                idx = np.random.choice(len(X_train), 50000, replace=False)
                X_train_sub = X_train[idx]
                y_train_sub = y_train[idx]
            else:
                X_train_sub = X_train
                y_train_sub = y_train
            X_train_seq = X_train_sub.reshape(X_train_sub.shape[0], X_train_sub.shape[1], 1)
            X_test_seq = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
            model = keras.Sequential([
                layers.Conv1D(filters=16, kernel_size=3, activation='relu', 
                             input_shape=(X_train.shape[1], 1), padding='same'),
                layers.MaxPooling1D(pool_size=2),
                layers.LSTM(16, return_sequences=True),
                layers.LSTM(8),
                layers.Dropout(0.3),
                layers.Dense(8, activation='relu'),
                layers.Dense(n_classes, activation='softmax')
            ])
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=0.001),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
            early_stop = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
            history = model.fit(
                X_train_seq, y_train_sub,
                validation_split=0.2,
                epochs=10,
                batch_size=CONFIG['deep_learning_batch_size'],
                callbacks=[early_stop],
                verbose=0
            )
            y_pred = np.argmax(model.predict(X_test_seq, verbose=0), axis=1)
            elapsed = time.time() - start_time
            self.training_times['cnn_lstm'] = elapsed
            return self._compute_metrics(y_test, y_pred, model, X_test)
        except Exception as e:
            return {'error': str(e)}
    
    def _compute_metrics(self, y_true, y_pred, model, X_test):
        try:
            metrics = {
                'accuracy': accuracy_score(y_true, y_pred),
                'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
                'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
                'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
                'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
                'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
                'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
                'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
                'matthews_corrcoef': matthews_corrcoef(y_true, y_pred),
                'n_samples': len(y_true)
            }
            metrics['classification_report'] = classification_report(y_true, y_pred, zero_division=0)
            metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred).tolist()
            return metrics
        except Exception as e:
            return {'error': str(e)}
    
    def run_all_baselines(self, X_train, y_train, X_test, y_test):
        print("\n" + "="*60)
        print("RUNNING SOTA BASELINE COMPARISONS")
        print("="*60)
        n_classes = len(np.unique(y_train))
        results = {}
        if XGB_AVAILABLE:
            results['xgboost'] = self.train_xgboost(X_train, y_train, X_test, y_test)
        results['naive_bayes'] = self.train_naive_bayes(X_train, y_train, X_test, y_test)
        if TF_AVAILABLE and n_classes > 1:
            results['dnn'] = self.train_dnn(X_train, y_train, X_test, y_test, n_classes)
            if X_train.shape[1] > 5:
                results['lstm'] = self.train_lstm(X_train, y_train, X_test, y_test, n_classes)
                results['cnn_lstm'] = self.train_cnn_lstm(X_train, y_train, X_test, y_test, n_classes)
        self.results = results
        return results

# ============================================================================
# ENHANCED TWO-STEP FRAMEWORK (unchanged except for Phase 1 assumption now valid)
# ============================================================================

class EnhancedTwoStepFramework:
    def __init__(self, model_type='random_forest', use_smote=True, smote_type='SMOTE', random_seed=42):
        self.model_type = model_type
        self.use_smote = use_smote
        self.smote_type = smote_type
        self.random_seed = random_seed
        self.filter_model = None
        self.ovr_models = {}
        self.classes = None
        self.class_distribution = None
        self.training_time = 0
        self.feature_importances = None
        self.error_analysis = {}
        self.imbalance_handler = AdvancedImbalanceHandler(technique=smote_type, random_seed=random_seed)
        
    def _create_model(self, class_weight=None):
        if self.model_type == 'random_forest':
            return RandomForestClassifier(
                n_estimators=200,
                max_depth=16,
                min_samples_split=4,
                min_samples_leaf=1,
                max_features='sqrt',
                class_weight=class_weight or 'balanced_subsample',
                random_state=self.random_seed,
                n_jobs=-1
            )
        else:
            return RandomForestClassifier(
                n_estimators=200,
                max_depth=16,
                max_features='sqrt',
                class_weight='balanced_subsample',
                random_state=self.random_seed,
                n_jobs=-1
            )
    
    def train_phase1_filter(self, X, y):
        print("🔒 Phase 1: Training Binary Filter...")
        start_time = time.time()
        # ✅ y is now guaranteed to have normal class = 0, so (y != 0) correctly
        # separates normal vs. anomalies.
        y_binary = (y != 0).astype(int)
        self.class_distribution = {
            'normal': int((y_binary == 0).sum()),
            'anomaly': int((y_binary == 1).sum())
        }
        print(f"   Normal: {self.class_distribution['normal']:,}, Anomaly: {self.class_distribution['anomaly']:,}")
        if self.class_distribution['anomaly'] == 0:
            print("   ⚠️ No anomaly samples!")
            class DummyModel:
                def predict(self, X):
                    return np.zeros(len(X))
                def predict_proba(self, X):
                    return np.ones((len(X), 2))
                def fit(self, X, y):
                    pass
            self.filter_model = DummyModel()
            return self.filter_model
        if self.use_smote:
            X_train_balanced, y_train_balanced = self.imbalance_handler.fit_resample(X, y_binary)
        else:
            X_train_balanced, y_train_balanced = X, y_binary
        self.filter_model = self._create_model(class_weight='balanced')
        self.filter_model.fit(X_train_balanced, y_train_balanced)
        y_pred = self.filter_model.predict(X)
        accuracy = accuracy_score(y_binary, y_pred)
        print(f"   Filter training accuracy: {accuracy:.4f}")
        elapsed = time.time() - start_time
        print(f"   ✅ Filter training took {elapsed:.2f}s")
        return self.filter_model
    
    def train_phase2_ovr_models(self, X, y):
        print("🎯 Phase 2: Training OvR Specialized Models...")
        start_time = time.time()
        self.classes = sorted(np.unique(y[y != 0]))
        print(f"   Attack classes: {len(self.classes)}")
        if len(self.classes) == 0:
            return self.ovr_models
        for attack_class in tqdm(self.classes, desc="OvR models"):
            y_ovr = (y == attack_class).astype(int)
            class_count = y_ovr.sum()
            total_count = len(y_ovr)
            if class_count > 0:
                if self.use_smote and class_count < total_count / 3:
                    try:
                        if class_count >= 2:
                            X_balanced, y_balanced = self.imbalance_handler.fit_resample(X, y_ovr)
                        else:
                            X_balanced, y_balanced = X, y_ovr
                    except:
                        X_balanced, y_balanced = X, y_ovr
                else:
                    X_balanced, y_balanced = X, y_ovr
                model = self._create_model(class_weight='balanced')
                model.fit(X_balanced, y_balanced)
                self.ovr_models[attack_class] = model
                print(f"   Class {attack_class}: {class_count:,} samples, IR={total_count/class_count:.1f}")
                if self.use_smote and class_count < total_count / 3:
                    del X_balanced, y_balanced
                    cleanup_memory()
        elapsed = time.time() - start_time
        self.training_time = elapsed
        print(f"   ✅ OvR training took {elapsed:.2f}s")
        return self.ovr_models
    
    def fit(self, X, y):
        print("🚀 Training Enhanced Two-Step Framework...")
        total_start = time.time()
        self.train_phase1_filter(X, y)
        self.train_phase2_ovr_models(X, y)
        if hasattr(self.filter_model, 'feature_importances_'):
            self.feature_importances = self.filter_model.feature_importances_
        total_time = time.time() - total_start
        print(f"✅ Total training time: {total_time:.2f}s")
        return self
    
    def predict(self, X):
        y_binary_pred = self.filter_model.predict(X)
        y_pred = np.zeros(len(X), dtype=int)
        anomaly_indices = np.where(y_binary_pred == 1)[0]
        if len(anomaly_indices) > 0 and len(self.ovr_models) > 0:
            X_anomaly = X.iloc[anomaly_indices] if isinstance(X, pd.DataFrame) else X[anomaly_indices]
            ovr_probs = {}
            for attack_class, model in self.ovr_models.items():
                try:
                    probs = model.predict_proba(X_anomaly)
                    if probs.shape[1] > 1:
                        ovr_probs[attack_class] = probs[:, 1]
                    else:
                        ovr_probs[attack_class] = model.predict(X_anomaly)
                except:
                    ovr_probs[attack_class] = model.predict(X_anomaly)
            for idx, orig_idx in enumerate(anomaly_indices):
                if len(ovr_probs) > 0:
                    probs = {cls: ovr_probs[cls][idx] for cls in self.ovr_models.keys()}
                    predicted_class = max(probs, key=probs.get)
                    y_pred[orig_idx] = predicted_class
        return y_pred
    
    def analyze_errors(self, X_test, y_test, y_pred):
        print("\n" + "="*60)
        print("🔍 DEEP ERROR ANALYSIS")
        print("="*60)
        errors = {}
        error_indices = np.where(y_test != y_pred)[0]
        if len(error_indices) == 0:
            print("   ✅ No errors found!")
            return errors
        print(f"   Total errors: {len(error_indices)}/{len(y_test)} ({len(error_indices)/len(y_test)*100:.2f}%)")
        for class_label in np.unique(y_test):
            class_mask = (y_test == class_label)
            class_errors = np.sum((y_test == class_label) & (y_test != y_pred))
            class_total = np.sum(class_mask)
            error_rate = class_errors / class_total if class_total > 0 else 0
            errors[f'class_{class_label}'] = {
                'total': int(class_total),
                'errors': int(class_errors),
                'error_rate': error_rate
            }
            print(f"   Class {class_label}: {class_errors}/{class_total} ({error_rate*100:.2f}%)")
        confusion = confusion_matrix(y_test, y_pred)
        error_confusions = []
        for i in range(len(confusion)):
            for j in range(len(confusion)):
                if i != j and confusion[i][j] > 0:
                    error_confusions.append((i, j, confusion[i][j]))
        error_confusions.sort(key=lambda x: x[2], reverse=True)
        errors['confusion_patterns'] = error_confusions[:5]
        print("\n   Most common error confusions:")
        for true_class, pred_class, count in error_confusions[:5]:
            print(f"      {true_class} → {pred_class}: {count} samples")
        self.error_analysis = errors
        return errors

# ============================================================================
# ENHANCED PIPELINE (unchanged)
# ============================================================================

class EnhancedNIDSPipeline:
    def __init__(self, config):
        self.config = config
        self.dataset_config = config.get('dataset_config', {})
        self.preprocessor = MemoryOptimizedDataPreprocessor(
            dataset_name=config.get('dataset_name', 'unknown'),
            leakage_threshold=config.get('leakage_threshold', 0.9)
        )
        self.framework = EnhancedTwoStepFramework(
            model_type=config.get('model_type', 'random_forest'),
            use_smote=config.get('use_smote', True),
            smote_type=config.get('smote_type', 'SMOTE'),
            random_seed=config.get('random_seed', 42)
        )
        self.sota_baselines = SOTABaselines(random_seed=config.get('random_seed', 42))
        self.results = {}
        self.total_time = 0
    
    def run_pipeline(self, df=None, label_col='Label'):
        print("=" * 80)
        print("🚀 ENHANCED AI-NIDS PIPELINE")
        print(f"📊 Dataset: {self.config.get('dataset_name', 'unknown')}")
        print(f"💾 Memory: {get_memory_usage():.2f} GB")
        print("=" * 80)
        pipeline_start = time.time()
        
        if df is None:
            df = self.preprocessor.load_data_from_kaggle(self.dataset_config)
            if df is None:
                return self.results
        
        print("\n" + "-"*60)
        print("STAGE 1: Data Preprocessing & Leakage Prevention")
        print("-"*60)
        X, y = self.preprocessor.preprocess(df, label_col)
        self.results['leakage_report'] = self.preprocessor.get_leakage_report()
        if len(X) == 0:
            return self.results
        
        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y,
                test_size=self.config.get('test_size', 0.2),
                random_state=self.config.get('random_seed', 42),
                stratify=y
            )
        except:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y,
                test_size=self.config.get('test_size', 0.2),
                random_state=self.config.get('random_seed', 42)
            )
        print(f"   Train: {len(X_train):,}, Test: {len(X_test):,}")
        print(f"   Classes: {len(np.unique(y_train))}")
        
        print("\n" + "-"*60)
        print("STAGE 2: Feature Selection")
        print("-"*60)
        n_features = min(self.config.get('mrmr_features', 20), X_train.shape[1])
        selector = SelectKBest(mutual_info_classif, k=n_features)
        X_train_selected = selector.fit_transform(X_train, y_train)
        X_test_selected = selector.transform(X_test)
        selected_features = X_train.columns[selector.get_support()].tolist()
        X_train_selected = pd.DataFrame(X_train_selected, columns=selected_features)
        X_test_selected = pd.DataFrame(X_test_selected, columns=selected_features)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_selected)
        X_test_scaled = scaler.transform(X_test_selected)
        X_train_scaled = pd.DataFrame(X_train_scaled, columns=selected_features)
        X_test_scaled = pd.DataFrame(X_test_scaled, columns=selected_features)
        self.results['selected_features'] = selected_features
        self.results['feature_count'] = len(selected_features)
        self.results['original_feature_count'] = X.shape[1]
        del X, X_train, X_test, X_train_selected, X_test_selected
        cleanup_memory()
        
        if len(X_train_scaled) > 100000:
            print(f"   ⚠️ Large dataset ({len(X_train_scaled):,}), using a shared sample for all models")
            sample_size = min(50000, len(X_train_scaled))
            idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)
            X_train_common = X_train_scaled.iloc[idx]
            y_train_common = y_train.iloc[idx]
            idx_test = np.random.choice(len(X_test_scaled), min(20000, len(X_test_scaled)), replace=False)
            X_test_common = X_test_scaled.iloc[idx_test]
            y_test_common = y_test.iloc[idx_test]
        else:
            X_train_common = X_train_scaled
            y_train_common = y_train
            X_test_common = X_test_scaled
            y_test_common = y_test

        print("\n" + "-"*60)
        print("STAGE 3: Two-Step Framework")
        print("-"*60)
        self.framework.fit(X_train_common, y_train_common)
        y_pred = self.framework.predict(X_test_common)
        framework_metrics = self._compute_metrics(y_test_common, y_pred)
        self.results['framework_metrics'] = framework_metrics
        if self.config.get('dataset_name') == 'UNSW-NB15':
            self.results['error_analysis'] = self.framework.analyze_errors(X_test_common, y_test_common, y_pred)

        print("\n" + "-"*60)
        print("STAGE 4: SOTA Baseline Comparisons (same evaluation set as the framework)")
        print("-"*60)
        sota_results = self.sota_baselines.run_all_baselines(
            X_train_common.values, y_train_common.values,
            X_test_common.values, y_test_common.values
        )
        self.results['sota_baselines'] = sota_results
        self.results['comparison'] = self._compile_comparison(framework_metrics, sota_results)
        
        self.total_time = time.time() - pipeline_start
        self.results['total_time'] = self.total_time
        print("\n" + "="*80)
        print("✅ PIPELINE COMPLETE")
        print("="*80)
        print(f"   Total time: {self.total_time:.2f}s")
        print(f"   Framework Macro-F1: {framework_metrics.get('f1_macro', 0):.4f}")
        if sota_results:
            best_f1 = 0
            best_model = None
            for name, res in sota_results.items():
                if 'error' not in res:
                    f1 = res.get('f1_macro', 0)
                    if f1 > best_f1:
                        best_f1 = f1
                        best_model = name
            if best_model:
                print(f"   Best SOTA: {best_model} ({best_f1:.4f})")
        print("="*80)
        return self.results
    
    def _compute_metrics(self, y_true, y_pred):
        try:
            return {
                'accuracy': accuracy_score(y_true, y_pred),
                'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
                'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
                'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
                'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
                'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
                'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
                'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
                'matthews_corrcoef': matthews_corrcoef(y_true, y_pred),
                'n_samples': len(y_true)
            }
        except Exception as e:
            return {'error': str(e)}
    
    def _compile_comparison(self, framework_metrics, sota_results):
        comparison = {
            'framework': {
                'f1_macro': framework_metrics.get('f1_macro', 0),
                'accuracy': framework_metrics.get('accuracy', 0)
            }
        }
        for name, results in sota_results.items():
            if 'error' not in results:
                comparison[name] = {
                    'f1_macro': results.get('f1_macro', 0),
                    'accuracy': results.get('accuracy', 0)
                }
        return comparison
    
    def export_results(self, output_path):
        print(f"\n📊 Exporting results to {output_path}...")
        json_path = output_path.replace('.csv', '.json')
        with open(json_path, 'w') as f:
            json.dump(self.results, f, default=str, indent=2)
        rows = []
        dataset_name = self.config.get('dataset_name', 'unknown')
        for metric, value in self.results.get('framework_metrics', {}).items():
            if isinstance(value, (int, float)):
                rows.append({
                    'model_type': 'framework',
                    'metric': metric,
                    'value': value,
                    'dataset': dataset_name,
                    'timestamp': datetime.now().isoformat()
                })
        for model_name, metrics in self.results.get('sota_baselines', {}).items():
            if isinstance(metrics, dict) and 'error' not in metrics:
                for metric, value in metrics.items():
                    if isinstance(value, (int, float)):
                        rows.append({
                            'model_type': model_name,
                            'metric': metric,
                            'value': value,
                            'dataset': dataset_name,
                            'timestamp': datetime.now().isoformat()
                        })
        for model_name, metrics in self.results.get('comparison', {}).items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    if isinstance(value, (int, float)):
                        rows.append({
                            'model_type': f'comparison_{model_name}',
                            'metric': metric,
                            'value': value,
                            'dataset': dataset_name,
                            'timestamp': datetime.now().isoformat()
                        })
        leakage = self.results.get('leakage_report', {})
        rows.append({
            'model_type': 'leakage',
            'metric': 'leaky_features_count',
            'value': leakage.get('leaky_count', 0),
            'dataset': dataset_name,
            'timestamp': datetime.now().isoformat()
        })
        df_results = pd.DataFrame(rows)
        df_results.to_csv(output_path, index=False)
        print(f"   ✅ Exported {len(df_results)} rows")
        return df_results

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def run_enhanced_pipeline_for_dataset(dataset_name, dataset_config):
    print(f"\n{'#'*80}")
    print(f"# DATASET: {dataset_name}")
    print(f"# Path: {dataset_config['path']}")
    print(f"{'#'*80}")
    config = CONFIG.copy()
    config['dataset_name'] = dataset_name
    config['dataset_config'] = dataset_config
    os.makedirs(config['output_dir'], exist_ok=True)
    pipeline = EnhancedNIDSPipeline(config)
    results = pipeline.run_pipeline()
    output_path = os.path.join(config['output_dir'], f'enhanced_nids_results_{dataset_name}.csv')
    pipeline.export_results(output_path)
    cleanup_memory()
    return pipeline, results

if __name__ == "__main__":
    print("="*80)
    print("MEMORY-OPTIMIZED ENHANCED AI-NIDS IMPLEMENTATION")
    print("With SOTA Comparisons & Deep Error Analysis")
    print("="*80)
    
    print("\n📂 Dataset Paths:")
    for name, config in DATASETS.items():
        status = "✅ EXISTS" if os.path.exists(config['path']) else "❌ NOT FOUND"
        print(f"  {name}: {config['path']}")
        print(f"    Status: {status}")
    
    available = [name for name, config in DATASETS.items() if os.path.exists(config['path'])]
    print(f"\n✅ Available datasets: {len(available)}")
    
    if available:
        all_results = {}
        for dataset_name in available:
            try:
                print(f"\n{'='*80}")
                print(f"PROCESSING: {dataset_name}")
                print(f"Memory before: {get_memory_usage():.2f} GB")
                print(f"{'='*80}")
                pipeline, results = run_enhanced_pipeline_for_dataset(
                    dataset_name, DATASETS[dataset_name]
                )
                all_results[dataset_name] = results
                print(f"\n✅ Completed: {dataset_name}")
                print(f"Memory after: {get_memory_usage():.2f} GB")
                cleanup_memory()
            except Exception as e:
                print(f"❌ Error on {dataset_name}: {e}")
                import traceback
                traceback.print_exc()
                cleanup_memory()
        
        if all_results:
            print("\n" + "="*80)
            print("📈 FINAL SUMMARY - ALL DATASETS")
            print("="*80)
            summary_data = []
            for dataset_name, results in all_results.items():
                framework_f1 = results.get('framework_metrics', {}).get('f1_macro', 0)
                framework_acc = results.get('framework_metrics', {}).get('accuracy', 0)
                best_sota = 0
                best_sota_name = 'N/A'
                sota_results = results.get('sota_baselines', {})
                for name, metrics in sota_results.items():
                    if 'error' not in metrics:
                        f1 = metrics.get('f1_macro', 0)
                        if f1 > best_sota:
                            best_sota = f1
                            best_sota_name = name
                leakage = results.get('leakage_report', {})
                summary_data.append({
                    'Dataset': dataset_name,
                    'Framework_F1': f'{framework_f1:.4f}',
                    'Framework_Acc': f'{framework_acc:.4f}',
                    'Best_SOTA': best_sota_name,
                    'Best_SOTA_F1': f'{best_sota:.4f}',
                    'F1_Diff': f'{best_sota - framework_f1:+.4f}',
                    'Leaky_Features': leakage.get('leaky_count', 0)
                })
            summary_df = pd.DataFrame(summary_data)
            summary_path = '/kaggle/working/results/sota_comparison_summary.csv'
            summary_df.to_csv(summary_path, index=False)
            print("\n" + summary_df.to_string(index=False))
            print(f"\n✅ Summary saved to: {summary_path}")
    
    print("\n" + "="*80)
    print("✅ EXECUTION COMPLETE")
    print("📊 Results saved to: /kaggle/working/results/")
    print("="*80)

✅ SHAP version: 0.51.0
MEMORY-OPTIMIZED ENHANCED AI-NIDS IMPLEMENTATION
With SOTA Comparisons & Deep Error Analysis

📂 Dataset Paths:
  NSL-KDD: /kaggle/input/datasets/karthikragavenderb/nsl-kdd-bert
    Status: ✅ EXISTS
  UNSW-NB15: /kaggle/input/datasets/likkisamarthreddy/unsw-nb15
    Status: ✅ EXISTS
  CIC-IDS2017: /kaggle/input/datasets/bertvankeulen/cicids-2017
    Status: ✅ EXISTS
  CSE-CICIDS2018: /kaggle/input/datasets/shrey213/cicids2018
    Status: ✅ EXISTS
  CIC-ToN-IoT: /kaggle/input/datasets/wahidulislambayazid/cic-ton-iot
    Status: ✅ EXISTS

✅ Available datasets: 5

PROCESSING: NSL-KDD
Memory before: 0.95 GB

################################################################################
# DATASET: NSL-KDD
# Path: /kaggle/input/datasets/karthikragavenderb/nsl-kdd-bert
################################################################################
🚀 ENHANCED AI-NIDS PIPELINE
📊 Dataset: NSL-KDD
💾 Memory: 0.95 GB
📂 Loading NSL-KDD...
   Found 1 files
   Found 1 matching

OvR models: 100%|██████████| 1/1 [00:02<00:00,  2.81s/it]

   Class 1: 25,069 samples, IR=2.0
   ✅ OvR training took 2.81s
✅ Total training time: 5.87s



------------------------------------------------------------
STAGE 4: SOTA Baseline Comparisons (same evaluation set as the framework)
------------------------------------------------------------

RUNNING SOTA BASELINE COMPARISONS
   Training XGBoost...
   Training Naive Bayes...
   Training DNN...


I0000 00:00:1787205389.280571      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1787205393.423599     122 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

   Memory after cleanup: 8.31 GB
   ✅ OvR training took 90.54s
✅ Total training time: 109.18s



------------------------------------------------------------
STAGE 4: SOTA Baseline Comparisons (same evaluation set as the framework)
------------------------------------------------------------

RUNNING SOTA BASELINE COMPARISONS
   Training XGBoost...
   Training Naive Bayes...
   Training DNN...
   Training LSTM...
   Training CNN-LSTM...

✅ PIPELINE COMPLETE
   Total time: 292.40s
   Framework Macro-F1: 0.7980
   Best SOTA: naive_bayes (0.6612)

📊 Exporting results to /kaggle/working/results/enhanced_nids_results_CSE-CICIDS2018.csv...
   ✅ Exported 61 rows
   Memory after cleanup: 7.86 GB

✅ Completed: CSE-CICIDS2018
Memory after: 7.86 GB
   Memory after cleanup: 7.86 GB

PROCESSING: CIC-ToN-IoT
Memory before: 7.86 GB

################################################################################
# DATASET: CIC-ToN-IoT
# Path: /kaggle/input/datasets/wahidulislambayazid/cic-ton-iot
################################################################################
🚀 ENHANCED AI-NIDS

OvR models: 100%|██████████| 1/1 [00:04<00:00,  4.57s/it]

   Class 1: 26,519 samples, IR=1.9
   ✅ OvR training took 4.58s
✅ Total training time: 9.51s



------------------------------------------------------------
STAGE 4: SOTA Baseline Comparisons (same evaluation set as the framework)
------------------------------------------------------------

RUNNING SOTA BASELINE COMPARISONS
   Training XGBoost...
   Training Naive Bayes...
   Training DNN...
   Training LSTM...
   Training CNN-LSTM...

✅ PIPELINE COMPLETE
   Total time: 259.12s
   Framework Macro-F1: 0.9934
   Best SOTA: xgboost (0.9935)

📊 Exporting results to /kaggle/working/results/enhanced_nids_results_CIC-ToN-IoT.csv...
   ✅ Exported 73 rows
   Memory after cleanup: 7.94 GB

✅ Completed: CIC-ToN-IoT
Memory after: 7.94 GB
   Memory after cleanup: 7.94 GB

📈 FINAL SUMMARY - ALL DATASETS

       Dataset Framework_F1 Framework_Acc   Best_SOTA Best_SOTA_F1 F1_Diff  Leaky_Features
       NSL-KDD       0.9945        0.9945     xgboost       0.9948 +0.0003               0
     UNSW-NB15       0.5582        0.8908     xgboost       0.6032 +0.0450               1
   CIC-IDS2017     